In [ ]:
import glob
import numpy as np
import os
import sys

# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

In [ ]:
dataset_name_folder = "dataset_sf9_512x33" 
input_layer = 512 * 33
layers = [input_layer, 1024, 256]  # compress X → 1024 → 256

GEENRATE_ = True  #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = [np.load(f) for f in files]
# Flatten each spectrogram to 1D (512*33 = X)
X = np.array([d.flatten() for d in data_list])  # shape: (num_samples, X)
print(X.shape)
################### Load all .npy files ########################################

multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)

if (GEENRATE_):
    
    weight_folder_path = "weight_16896_1024_256_more"

    # Check if folder exists, if not create it
    if not os.path.exists(weight_folder_path):
        os.makedirs(weight_folder_path)
        print(f"Folder created: {weight_folder_path}")
    else:
        print(f"Folder already exists: {weight_folder_path}")
   
    layer_losses = multi_bam.train(X, num_epochs=8, batch_size=64)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{weight_folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(): 
    ## EXAMPLE HOW TO LOAD WEIGHT
    layers = [512*33, 1024, 256] # <-- must match training

    multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)
    for i, bam in enumerate(multi_bam.bams):
        w_np = np.load(f"weight_16896_1024_256/weights_layer_{i}.npy")
        bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
